# GameForge AI — Stage 4: Neural FlowSTN Deformation Experiment
### Reproducible Kaggle Workflow for 2D Character Pose Deformation

This notebook trains and evaluates the **FlowSTN (Dense Flow Spatial Transformer Network)** on the GameForge AI synthetic paired-pose dataset.

#### Learning Task:
- **Input**: Source sprite (128x128x4 RGBA) + 32-dim pose delta vector (16 joints * [dx, dy]).
- **Prediction**: Continuous 2D coordinate displacement field $\mathbf{F}(x, y) = (\Delta x, \Delta y)$.
- **Warp**: Differentiable bilinear grid sampling $\hat{I}_{tgt}(x, y) = I_{src}(x + \Delta x, y + \Delta y)$.
- **Ground Truth**: Held-out character identities deformed to standard target poses.

#### What to Upload to Kaggle:
- Upload only the `synthetic_poses` directory (containing `manifest.json`, `sources/`, `targets/` — approx 1.5 MB) as a Kaggle Dataset named `gameforge-synthetic-poses`.
- No API keys, credentials, or full repository uploads required.

In [ ]:
# Cell 1: Environment & Hardware Acceleration Check
import os
import sys
import json
import time
import random
import zipfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")

# Check for GPU accelerator
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUCCESS: Hardware Acceleration Active! Found {len(gpus)} GPU(s): {gpus}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"Memory growth setting note: {e}")
else:
    print("WARNING: No GPU detected. Running on CPU. Training will take slightly longer.")

# Set deterministic random seeds
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Deterministic random seeds set to 42.")

In [ ]:
# Cell 2: Dataset Discovery, Manifest Integrity & Validation
POSSIBLE_DATASET_PATHS = [
    "/kaggle/input/gameforge-synthetic-poses",
    "/kaggle/input/synthetic-poses",
    "/kaggle/input/gameforge-synthetic-poses/synthetic_poses",
    "./data/synthetic_poses",
    "../backend/data/synthetic_poses",
    "./backend/data/synthetic_poses"
]

DATASET_DIR = None
for path in POSSIBLE_DATASET_PATHS:
    manifest_candidate = os.path.join(path, "manifest.json")
    if os.path.exists(manifest_candidate):
        DATASET_DIR = os.path.abspath(path)
        print(f"Found valid dataset directory: {DATASET_DIR}")
        break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not find 'manifest.json' in any standard Kaggle input location.\n"
        "Please ensure your Kaggle dataset contains 'manifest.json' and is attached to the notebook."
    )

# Validate manifest integrity
manifest_path = os.path.join(DATASET_DIR, "manifest.json")
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

samples = manifest.get("samples", [])
print(f"Total manifest samples: {len(samples)}")

# Verify file existence & character identity separation
train_chars, val_chars, test_chars = set(), set(), set()
missing_files = 0

for s in samples:
    src = os.path.join(DATASET_DIR, s["source_image_path"])
    tgt = os.path.join(DATASET_DIR, s["target_image_path"])
    if not os.path.exists(src) or not os.path.exists(tgt):
        missing_files += 1
    
    char_id = s.get("character_id", s["sample_id"].split("_")[0])
    split = s["split"]
    if split == "train":
        train_chars.add(char_id)
    elif split == "val":
        val_chars.add(char_id)
    elif split == "test":
        test_chars.add(char_id)

print(f"Missing files count: {missing_files}")
print(f"Train characters ({len(train_chars)}): {sorted(list(train_chars))[:5]}...")
print(f"Val characters   ({len(val_chars)}): {sorted(list(val_chars))}")
print(f"Test characters  ({len(test_chars)}): {sorted(list(test_chars))}")

# Disjointness check
leakage_train_val = train_chars.intersection(val_chars)
leakage_train_test = train_chars.intersection(test_chars)
leakage_val_test = val_chars.intersection(test_chars)

assert len(leakage_train_val) == 0, f"Data leakage! Train and Val overlap on {leakage_train_val}"
assert len(leakage_train_test) == 0, f"Data leakage! Train and Test overlap on {leakage_train_test}"
assert len(leakage_val_test) == 0, f"Data leakage! Val and Test overlap on {leakage_val_test}"
assert missing_files == 0, f"Dataset has {missing_files} missing image files!"
print("DATASET INTEGRITY AUDIT PASSED: 100% disjoint character splits, 0 missing files.")

In [ ]:
# Cell 3: Data Loader & Vector Extraction
JOINT_NAMES = [
    "head", "neck", "shoulder_left", "shoulder_right",
    "elbow_left", "elbow_right", "wrist_left", "wrist_right",
    "pelvis", "hip_left", "hip_right",
    "knee_left", "knee_right", "ankle_left", "ankle_right", "root"
]

def extract_pose_delta(src_skel, tgt_skel):
    sj = src_skel.get("joints", {})
    tj = tgt_skel.get("joints", {})
    deltas = []
    for name in JOINT_NAMES:
        s_pt = sj.get(name, {})
        t_pt = tj.get(name, {})
        deltas.append(t_pt.get("x", 0.5) - s_pt.get("x", 0.5))
        deltas.append(t_pt.get("y", 0.5) - s_pt.get("y", 0.5))
    return np.array(deltas, dtype=np.float32)

def load_split_arrays(manifest, dataset_dir, split_name):
    split_samples = [s for s in manifest["samples"] if s["split"] == split_name]
    x_imgs, x_deltas, y_targets = [], [], []
    for s in split_samples:
        src_p = os.path.join(dataset_dir, s["source_image_path"])
        tgt_p = os.path.join(dataset_dir, s["target_image_path"])
        
        src_img = np.array(Image.open(src_p).convert("RGBA"), dtype=np.float32) / 255.0
        tgt_img = np.array(Image.open(tgt_p).convert("RGBA"), dtype=np.float32) / 255.0
        delta = extract_pose_delta(s["source_skeleton"], s["target_skeleton"])
        
        x_imgs.append(src_img)
        x_deltas.append(delta)
        y_targets.append(tgt_img)
    return np.array(x_imgs, dtype=np.float32), np.array(x_deltas, dtype=np.float32), np.array(y_targets, dtype=np.float32)

print("Loading dataset arrays into memory...")
X_train_img, X_train_delta, Y_train_tgt = load_split_arrays(manifest, DATASET_DIR, "train")
X_val_img, X_val_delta, Y_val_tgt = load_split_arrays(manifest, DATASET_DIR, "val")
X_test_img, X_test_delta, Y_test_tgt = load_split_arrays(manifest, DATASET_DIR, "test")

print(f"Train set: {X_train_img.shape[0]} pairs (Images: {X_train_img.shape}, Deltas: {X_train_delta.shape})")
print(f"Val set:   {X_val_img.shape[0]} pairs")
print(f"Test set:  {X_test_img.shape[0]} pairs (Held out unseen characters)")

In [ ]:
# Cell 4: FlowSTN Model Architecture Definition
@tf.keras.utils.register_keras_serializable()
class BilinearGridWarp(layers.Layer):
    """Differentiable 2D bilinear grid sampling layer."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        image, flow = inputs
        shape = tf.shape(image)
        batch_size, height, width = shape[0], shape[1], shape[2]
        h_f = tf.cast(height, tf.float32)
        w_f = tf.cast(width, tf.float32)

        x_lin = tf.linspace(0.0, 1.0, width)
        y_lin = tf.linspace(0.0, 1.0, height)
        grid_x, grid_y = tf.meshgrid(x_lin, y_lin)
        base_grid = tf.expand_dims(tf.stack([grid_x, grid_y], axis=-1), axis=0)

        sample_grid = base_grid + flow
        sample_x = tf.clip_by_value(sample_grid[..., 0] * (w_f - 1.0), 0.0, w_f - 1.0)
        sample_y = tf.clip_by_value(sample_grid[..., 1] * (h_f - 1.0), 0.0, h_f - 1.0)

        x0 = tf.floor(sample_x)
        x1 = tf.minimum(x0 + 1.0, w_f - 1.0)
        y0 = tf.floor(sample_y)
        y1 = tf.minimum(y0 + 1.0, h_f - 1.0)

        wa = tf.expand_dims((x1 - sample_x) * (y1 - sample_y), axis=-1)
        wb = tf.expand_dims((x1 - sample_x) * (sample_y - y0), axis=-1)
        wc = tf.expand_dims((sample_x - x0) * (y1 - sample_y), axis=-1)
        wd = tf.expand_dims((sample_x - x0) * (sample_y - y0), axis=-1)

        idx_x0 = tf.cast(x0, tf.int32)
        idx_x1 = tf.cast(x1, tf.int32)
        idx_y0 = tf.cast(y0, tf.int32)
        idx_y1 = tf.cast(y1, tf.int32)

        batch_idx = tf.broadcast_to(tf.range(batch_size)[:, tf.newaxis, tf.newaxis], [batch_size, height, width])

        def gather_channel(y_idx, x_idx):
            indices = tf.stack([batch_idx, y_idx, x_idx], axis=-1)
            return tf.gather_nd(image, indices)

        Ia = gather_channel(idx_y0, idx_x0)
        Ib = gather_channel(idx_y1, idx_x0)
        Ic = gather_channel(idx_y0, idx_x1)
        Id = gather_channel(idx_y1, idx_x1)

        warped = wa * Ia + wb * Ib + wc * Ic + wd * Id
        return tf.clip_by_value(warped, 0.0, 1.0)

@tf.keras.utils.register_keras_serializable()
class FlowScalingLayer(layers.Layer):
    """Scales raw tanh flow field to maximum normalized displacement."""
    def __init__(self, max_displacement: float = 0.25, **kwargs):
        super().__init__(**kwargs)
        self.max_displacement = max_displacement

    def call(self, inputs):
        return inputs * self.max_displacement

    def get_config(self):
        config = super().get_config()
        config.update({"max_displacement": self.max_displacement})
        return config

def build_flow_stn(image_shape=(128, 128, 4), pose_dim=32, max_displacement=0.25):
    src_input = layers.Input(shape=image_shape, name="source_image")
    pose_input = layers.Input(shape=(pose_dim,), name="pose_delta")

    # Pose pathway
    d1 = layers.Dense(64, activation="relu")(pose_input)
    d2 = layers.Dense(16 * 16 * 16, activation="relu")(d1)
    d_spatial = layers.Reshape((16, 16, 16))(d2)
    d_up = layers.UpSampling2D(size=(8, 8), interpolation="bilinear")(d_spatial)

    # Visual feature pathway
    c1 = layers.Conv2D(16, (3, 3), padding="same", activation="relu")(src_input)
    c2 = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(c1)

    # Fusion & Flow Head
    merged = layers.Concatenate()([c2, d_up])
    f1 = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(merged)
    f2 = layers.Conv2D(16, (3, 3), padding="same", activation="relu")(f1)
    raw_flow = layers.Conv2D(2, (3, 3), padding="same", activation="tanh")(f2)
    flow_field = FlowScalingLayer(max_displacement=max_displacement, name="flow_field")(raw_flow)

    # Warping
    warped_output = BilinearGridWarp(name="bilinear_warp")([src_input, flow_field])
    return Model(inputs=[src_input, pose_input], outputs=[warped_output, flow_field], name="FlowSTN")

model = build_flow_stn()
model.summary()
print(f"FlowSTN constructed successfully: {model.count_params():,} trainable parameters.")

In [ ]:
# Cell 5: Controlled Training Loop with Multi-Objective Loss
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./kaggle_output"
os.makedirs(WORKING_DIR, exist_ok=True)

EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
TV_WEIGHT = 0.01  # Flow smoothness regularization

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_img, X_train_delta, Y_train_tgt)).shuffle(128, seed=SEED).batch(BATCH_SIZE)

@tf.function
def train_step(src_b, delta_b, tgt_b):
    with tf.GradientTape() as tape:
        pred_img, flow = model([src_b, delta_b], training=True)
        l1_loss = tf.reduce_mean(tf.abs(pred_img - tgt_b))
        tv_loss = tf.reduce_mean(tf.image.total_variation(flow))
        total_loss = l1_loss + TV_WEIGHT * tv_loss
    grads = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total_loss, l1_loss, tv_loss

@tf.function
def val_step(src_b, delta_b, tgt_b):
    pred_img, flow = model([src_b, delta_b], training=False)
    l1_loss = tf.reduce_mean(tf.abs(pred_img - tgt_b))
    mse_loss = tf.reduce_mean(tf.square(pred_img - tgt_b))
    return l1_loss, mse_loss

history = {"epoch": [], "train_loss": [], "train_l1": [], "val_l1": [], "val_mse": []}
best_val_l1 = float("inf")
best_checkpoint_path = os.path.join(WORKING_DIR, "flow_stn_model.keras")

print(f"Starting training for {EPOCHS} epochs...")
start_training_time = time.time()

for epoch in range(1, EPOCHS + 1):
    t_epoch_start = time.time()
    train_losses, train_l1s, train_tvs = [], [], []
    for src_b, delta_b, tgt_b in train_dataset:
        tot, l1, tv = train_step(src_b, delta_b, tgt_b)
        train_losses.append(tot.numpy())
        train_l1s.append(l1.numpy())
        train_tvs.append(tv.numpy())
        
    val_l1, val_mse = val_step(X_val_img, X_val_delta, Y_val_tgt)
    val_l1 = float(val_l1.numpy())
    val_mse = float(val_mse.numpy())
    avg_train_loss = float(np.mean(train_losses))
    avg_train_l1 = float(np.mean(train_l1s))
    
    history["epoch"].append(epoch)
    history["train_loss"].append(avg_train_loss)
    history["train_l1"].append(avg_train_l1)
    history["val_l1"].append(val_l1)
    history["val_mse"].append(val_mse)
    
    epoch_time = time.time() - t_epoch_start
    print(f"Epoch {epoch:2d}/{EPOCHS} [{epoch_time:.2f}s] - Train Loss: {avg_train_loss:.4f} (L1: {avg_train_l1:.4f}) | Val L1: {val_l1:.4f} | Val MSE: {val_mse:.6f}")
    
    if val_l1 < best_val_l1:
        best_val_l1 = val_l1
        model.save(best_checkpoint_path)
        print(f"  --> Saved new best checkpoint (Val L1: {best_val_l1:.4f}) to {best_checkpoint_path}")

total_train_sec = time.time() - start_training_time
print(f"Training complete in {total_train_sec:.2f}s! Best Val L1: {best_val_l1:.4f}")

In [ ]:
# Cell 6: Rigorous Evaluation on Held-Out Test Split (7 Unseen Characters)
print("Evaluating best checkpoint on held-out test split...")
best_model = tf.keras.models.load_model(
    best_checkpoint_path,
    custom_objects={"BilinearGridWarp": BilinearGridWarp, "FlowScalingLayer": FlowScalingLayer}
)

# Inference timing
t0 = time.perf_counter()
test_preds, test_flows = best_model.predict([X_test_img, X_test_delta], batch_size=8, verbose=0)
inf_time = (time.perf_counter() - t0) * 1000.0 / len(X_test_img)

# Metric calculations
test_l1 = float(np.mean(np.abs(test_preds - Y_test_tgt)))
test_mse = float(np.mean(np.square(test_preds - Y_test_tgt)))
test_psnr = float(10.0 * np.log10(1.0 / (test_mse + 1e-10)))

# Alpha integrity & Silhouette IoU
pred_alpha = test_preds[..., 3]
tgt_alpha = Y_test_tgt[..., 3]

pred_mask = pred_alpha > 0.1
tgt_mask = tgt_alpha > 0.1

intersection = np.logical_and(pred_mask, tgt_mask).sum(axis=(1, 2))
union = np.logical_or(pred_mask, tgt_mask).sum(axis=(1, 2))
target_ious = [float(i / u) if u > 0 else 1.0 for i, u in zip(intersection, union)]
mean_target_iou = float(np.mean(target_ious))

pred_mass = np.sum(pred_alpha, axis=(1, 2))
tgt_mass = np.sum(tgt_alpha, axis=(1, 2))
alpha_integrities = [float(min(1.0, p / (t + 1e-8))) * 100.0 for p, t in zip(pred_mass, tgt_mass)]
mean_alpha_integrity = float(np.mean(alpha_integrities))

eval_summary = {
    "test_sample_count": len(X_test_img),
    "l1_reconstruction_error": round(test_l1, 4),
    "mse": round(test_mse, 6),
    "psnr_db": round(test_psnr, 2),
    "target_silhouette_iou": round(mean_target_iou, 4),
    "alpha_integrity_pct": round(mean_alpha_integrity, 2),
    "latency_ms_per_frame": round(inf_time, 2)
}

print("\n================ HELD-OUT TEST EVALUATION RESULTS ================")
print(json.dumps(eval_summary, indent=2))

# Save summary and training history to disk
with open(os.path.join(WORKING_DIR, "evaluation_summary.json"), "w") as f:
    json.dump(eval_summary, f, indent=2)

with open(os.path.join(WORKING_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# Cell 7: Visual Verification Grid Generation
num_viz = min(4, len(X_test_img))
fig, axes = plt.subplots(num_viz, 4, figsize=(14, 3.5 * num_viz))

for i in range(num_viz):
    # Source
    axes[i, 0].imshow(X_test_img[i])
    axes[i, 0].set_title(f"Test #{i+1} Source")
    axes[i, 0].axis('off')
    
    # Flow Magnitude
    flow_mag = np.linalg.norm(test_flows[i], axis=-1)
    axes[i, 1].imshow(flow_mag, cmap='magma')
    axes[i, 1].set_title(f"Flow Magnitude (Max: {flow_mag.max():.3f})")
    axes[i, 1].axis('off')
    
    # Predicted Output
    axes[i, 2].imshow(test_preds[i])
    axes[i, 2].set_title(f"FlowSTN Deformed (IoU: {target_ious[i]:.2f})")
    axes[i, 2].axis('off')
    
    # Ground Truth Target
    axes[i, 3].imshow(Y_test_tgt[i])
    axes[i, 3].set_title("Ground Truth Target")
    axes[i, 3].axis('off')

plt.tight_layout()
grid_path = os.path.join(WORKING_DIR, "test_evaluation_grid.png")
plt.savefig(grid_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved visual evaluation grid to {grid_path}")

In [ ]:
# Cell 8: Compact Artifact Packaging for Kaggle Download
zip_output_path = os.path.join(WORKING_DIR, "gameforge_flow_stn_artifacts.zip")

files_to_pack = [
    ("flow_stn_model.keras", os.path.join(WORKING_DIR, "flow_stn_model.keras")),
    ("evaluation_summary.json", os.path.join(WORKING_DIR, "evaluation_summary.json")),
    ("training_history.json", os.path.join(WORKING_DIR, "training_history.json")),
    ("test_evaluation_grid.png", os.path.join(WORKING_DIR, "test_evaluation_grid.png"))
]

with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for arcname, fpath in files_to_pack:
        if os.path.exists(fpath):
            zipf.write(fpath, arcname=arcname)
            size_kb = os.path.getsize(fpath) / 1024.0
            print(f"Packed: {arcname} ({size_kb:.1f} KB)")

total_zip_mb = os.path.getsize(zip_output_path) / (1024.0 * 1024.0)
print(f"\nSUCCESS: Artifact package created at {zip_output_path} ({total_zip_mb:.2f} MB).")
print("Download this zip file from the Kaggle 'Output' tab.")

### Downloading & Integrating into GameForge AI:
1. Download `gameforge_flow_stn_artifacts.zip` from Kaggle's **Output** panel.
2. Extract `flow_stn_model.keras` to your local repository under `models/deformation_neural/flow_stn_model.keras`.
3. In GameForge AI, the neural deformation adapter will automatically detect the checkpoint and allow selecting **Neural STN** alongside the **Geometric LBS Baseline**.